# 04 — Churn Labeling

**Nhiệm vụ:** tạo observation theo từng lần mua thực tế, xác định lần mua kế tiếp, gán nhãn `churn_90`, `churn_120`, `churn_180`, kiểm soát right-censoring và lưu tập snapshot có nhãn. Notebook này **không tạo feature cho mô hình**.


## Cell 01 — Khởi tạo môi trường và đường dẫn


In [1]:
%pip -q install pyspark

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("OlistChurnLabeling").getOrCreate()

data_dir = Path("/content/drive/MyDrive/BigData/DA-BigData/code/data")
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

orders_prepared_path = processed_dir / "olist_orders_prepared.csv"
orders_raw_path = data_dir / "olist_orders_dataset.csv"
customers_path = data_dir / "olist_customers_dataset.csv"

if not customers_path.exists():
    raise FileNotFoundError(f"Không tìm thấy: {customers_path}")

if not orders_prepared_path.exists() and not orders_raw_path.exists():
    raise FileNotFoundError(
        "Không tìm thấy cả olist_orders_prepared.csv và olist_orders_dataset.csv"
    )

orders_path = orders_prepared_path if orders_prepared_path.exists() else orders_raw_path

print("Data directory     :", data_dir)
print("Processed directory:", processed_dir)
print("Orders source      :", orders_path)


Mounted at /content/drive
Data directory     : /content/drive/MyDrive/BigData/DA-BigData/code/data
Processed directory: /content/drive/MyDrive/BigData/DA-BigData/code/data/processed
Orders source      : /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_orders_prepared.csv


## Cell 02 — Đọc đơn hàng đã chuẩn bị và tạo chuỗi mua hợp lệ


In [2]:
from pyspark.sql import functions as F

orders = (
    spark.read.option("header", True)
    .csv(str(orders_path))
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

customers = (
    spark.read.option("header", True)
    .csv(str(customers_path))
    .select("customer_id", "customer_unique_id")
)

if customers.count() != customers.select("customer_id").distinct().count():
    raise ValueError("customer_id bị trùng trong bảng customers")

if orders.count() != orders.select("order_id").distinct().count():
    raise ValueError("order_id bị trùng trong bảng orders")

purchases = (
    orders
    .filter(F.col("order_status") == "delivered")
    .join(customers, "customer_id", "inner")
    .withColumn("purchase_at", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("delivered_at", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("estimated_delivery_at", F.to_timestamp("order_estimated_delivery_date"))
    .filter(
        F.col("customer_unique_id").isNotNull()
        & F.col("order_id").isNotNull()
        & F.col("purchase_at").isNotNull()
    )
    .select(
        "customer_unique_id",
        "order_id",
        "purchase_at",
        "delivered_at",
        "estimated_delivery_at"
    )
    .dropDuplicates(["order_id"])
    .cache()
)

purchase_rows = purchases.count()
unique_customers = purchases.select("customer_unique_id").distinct().count()
data_end_at = purchases.agg(F.max("purchase_at").alias("data_end_at")).first()["data_end_at"]

if data_end_at is None:
    raise ValueError("Không tìm thấy purchase_at hợp lệ")

print("Số đơn delivered hợp lệ:", purchase_rows)
print("Số customer_unique_id   :", unique_customers)
print("Ngày mua cuối dữ liệu   :", data_end_at)


Số đơn delivered hợp lệ: 96478
Số customer_unique_id   : 93358
Ngày mua cuối dữ liệu   : 2018-08-29 15:00:37


## Cell 03 — Tạo purchase-event snapshot và gán nhãn churn 90/120/180 ngày


In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# 1. SẮP XẾP CHUỖI MUA THEO CUSTOMER_UNIQUE_ID


w_order = (
    Window
    .partitionBy("customer_unique_id")
    .orderBy(F.col("purchase_at"), F.col("order_id"))
)

w_history = (
    Window
    .partitionBy("customer_unique_id")
    .orderBy(F.col("purchase_at"), F.col("order_id"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

purchase_events = (
    purchases
    .withColumn("purchase_sequence", F.row_number().over(w_order))
    .withColumn("previous_purchase_at", F.lag("purchase_at").over(w_order))
    .withColumn("next_purchase_at", F.lead("purchase_at").over(w_order))
    .withColumn("first_purchase_at", F.first("purchase_at").over(w_history))
    .withColumn(
        "days_since_previous_purchase",
        F.when(
            F.col("previous_purchase_at").isNotNull(),
            (
                F.unix_timestamp("purchase_at")
                - F.unix_timestamp("previous_purchase_at")
            ) / 86400.0
        )
    )
    .withColumn(
        "gap_days",
        F.when(
            F.col("next_purchase_at").isNotNull(),
            (
                F.unix_timestamp("next_purchase_at")
                - F.unix_timestamp("purchase_at")
            ) / 86400.0
        )
    )
    .withColumn(
        "customer_tenure_days",
        (
            F.unix_timestamp("purchase_at")
            - F.unix_timestamp("first_purchase_at")
        ) / 86400.0
    )
    .withColumn(
        "avg_interpurchase_days_to_snapshot",
        F.avg("days_since_previous_purchase").over(w_history)
    )
    .withColumn(
        "max_interpurchase_days_to_snapshot",
        F.max("days_since_previous_purchase").over(w_history)
    )
    .withColumnRenamed("order_id", "snapshot_order_id")
    .withColumnRenamed("purchase_at", "snapshot_at")
    .withColumn("snapshot_date", F.to_date("snapshot_at"))
)

# 2. FOLLOW-UP VÀ NHÃN CHURN


DATA_END = F.lit(data_end_at).cast("timestamp")

for threshold in [90, 120, 180]:
    followup_col = f"followup_end_{threshold}"
    churn_col = f"churn_{threshold}"
    status_col = f"label_status_{threshold}"
    full_col = f"has_full_followup_{threshold}"

    purchase_events = (
        purchase_events
        .withColumn(
            followup_col,
            F.expr(f"snapshot_at + INTERVAL {threshold} DAYS")
        )
        .withColumn(
            full_col,
            F.when(F.col(followup_col) <= DATA_END, 1).otherwise(0)
        )
        .withColumn(
            churn_col,
            F.when(
                F.col("next_purchase_at").isNotNull()
                & (F.col("next_purchase_at") <= F.col(followup_col)),
                F.lit(0)
            )
            .when(
                F.col("next_purchase_at").isNotNull()
                & (F.col("next_purchase_at") > F.col(followup_col)),
                F.lit(1)
            )
            .when(
                F.col("next_purchase_at").isNull()
                & (F.col(full_col) == 1),
                F.lit(1)
            )
            .otherwise(F.lit(None).cast("int"))
        )
        .withColumn(
            status_col,
            F.when(F.col(churn_col) == 0, F.lit("NON_CHURN"))
            .when(
                (F.col(churn_col) == 1)
                & F.col("next_purchase_at").isNotNull(),
                F.lit(f"CHURN_GAP_GT_{threshold}")
            )
            .when(
                (F.col(churn_col) == 1)
                & F.col("next_purchase_at").isNull(),
                F.lit("CHURN_NO_RETURN")
            )
            .otherwise(F.lit("CENSORED"))
        )
    )

purchase_events = (
    purchase_events
    .withColumn(
        "followup_days_available",
        (
            F.unix_timestamp(DATA_END)
            - F.unix_timestamp("snapshot_at")
        ) / 86400.0
    )
    .cache()
)


# 3. KIỂM TRA LOGIC NHÃN


duplicate_event_keys = (
    purchase_events
    .groupBy("customer_unique_id", "snapshot_order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_sequence = (
    purchase_events
    .filter(
        F.col("next_purchase_at").isNotNull()
        & (F.col("next_purchase_at") < F.col("snapshot_at"))
    )
    .count()
)

invalid_180 = (
    purchase_events
    .filter(
        (
            (F.col("churn_180") == 0)
            & (
                F.col("next_purchase_at").isNull()
                | (F.col("next_purchase_at") > F.col("followup_end_180"))
            )
        )
        | (
            (F.col("churn_180") == 1)
            & F.col("next_purchase_at").isNull()
            & (F.col("has_full_followup_180") != 1)
        )
    )
    .count()
)

invalid_label_order = (
    purchase_events
    .filter(
        F.col("churn_90").isNotNull()
        & F.col("churn_120").isNotNull()
        & F.col("churn_180").isNotNull()
        & (
            (F.col("churn_90") < F.col("churn_120"))
            | (F.col("churn_120") < F.col("churn_180"))
        )
    )
    .count()
)

print("Duplicate customer + snapshot_order_id:", duplicate_event_keys)
print("next_purchase_at < snapshot_at         :", invalid_sequence)
print("Nhãn churn_180 không hợp lệ            :", invalid_180)
print("Sai thứ tự churn_90/120/180            :", invalid_label_order)

if any(x > 0 for x in [duplicate_event_keys, invalid_sequence, invalid_180, invalid_label_order]):
    raise ValueError("Kiểm tra logic snapshot/nhãn chưa đạt")

# 4. LƯU TOÀN BỘ PURCHASE-EVENT SNAPSHOT, KỂ CẢ CENSORED


snapshot_all_dir = processed_dir / "olist_customer_purchase_event_snapshots"

(
    purchase_events
    .repartition(8, "snapshot_date")
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(snapshot_all_dir))
)

print("Đã lưu toàn bộ event snapshot:", snapshot_all_dir)


Duplicate customer + snapshot_order_id: 0
next_purchase_at < snapshot_at         : 0
Nhãn churn_180 không hợp lệ            : 0
Sai thứ tự churn_90/120/180            : 0
Đã lưu toàn bộ event snapshot: /content/drive/MyDrive/BigData/DA-BigData/code/data/processed/olist_customer_purchase_event_snapshots


## Cell 04 — Kiểm tra nhãn, right-censoring và lưu tập có nhãn 180 ngày


In [5]:
from pyspark.sql import functions as F

snapshot_all_dir = processed_dir / "olist_customer_purchase_event_snapshots"
snapshot_labeled_dir = processed_dir / "olist_customer_purchase_event_snapshots_labeled_180"

snapshots = (
    spark.read.option("header", True)
    .csv(str(snapshot_all_dir))
    .withColumn("snapshot_at", F.to_timestamp("snapshot_at"))
    .withColumn("snapshot_date", F.to_date("snapshot_date"))
    .withColumn("previous_purchase_at", F.to_timestamp("previous_purchase_at"))
    .withColumn("next_purchase_at", F.to_timestamp("next_purchase_at"))
    .withColumn("followup_end_90", F.to_timestamp("followup_end_90"))
    .withColumn("followup_end_120", F.to_timestamp("followup_end_120"))
    .withColumn("followup_end_180", F.to_timestamp("followup_end_180"))
    .withColumn("purchase_sequence", F.col("purchase_sequence").cast("int"))
    .withColumn("days_since_previous_purchase", F.col("days_since_previous_purchase").cast("double"))
    .withColumn("gap_days", F.col("gap_days").cast("double"))
    .withColumn("customer_tenure_days", F.col("customer_tenure_days").cast("double"))
    .withColumn("avg_interpurchase_days_to_snapshot", F.col("avg_interpurchase_days_to_snapshot").cast("double"))
    .withColumn("max_interpurchase_days_to_snapshot", F.col("max_interpurchase_days_to_snapshot").cast("double"))
    .withColumn("followup_days_available", F.col("followup_days_available").cast("double"))
    .withColumn("churn_90", F.col("churn_90").cast("int"))
    .withColumn("churn_120", F.col("churn_120").cast("int"))
    .withColumn("churn_180", F.col("churn_180").cast("int"))
    .cache()
)

# 1. TỔNG QUAN EVENT SNAPSHOT


summary = (
    snapshots
    .agg(
        F.count("*").alias("so_event_snapshot"),
        F.countDistinct("customer_unique_id").alias("so_khach"),
        F.sum(F.when(F.col("churn_180") == 0, 1).otherwise(0)).alias("non_churn_180"),
        F.sum(F.when(F.col("churn_180") == 1, 1).otherwise(0)).alias("churn_180"),
        F.sum(F.when(F.col("churn_180").isNull(), 1).otherwise(0)).alias("censored_180")
    )
)

print("TỔNG QUAN PURCHASE-EVENT SNAPSHOT")
summary.show(truncate=False)

print("TRẠNG THÁI NHÃN 180 NGÀY")
(
    snapshots
    .groupBy("label_status_180")
    .agg(
        F.count("*").alias("so_event"),
        F.countDistinct("customer_unique_id").alias("so_khach")
    )
    .orderBy("label_status_180")
    .show(truncate=False)
)


# 2. SỐ EVENT TRÊN MỖI KHÁCH


events_per_customer = (
    snapshots
    .groupBy("customer_unique_id")
    .agg(F.count("*").alias("so_lan_mua"))
)

print("PHÂN BỐ SỐ LẦN MUA TRÊN MỖI KHÁCH")
(
    events_per_customer
    .groupBy("so_lan_mua")
    .agg(F.count("*").alias("so_khach"))
    .orderBy("so_lan_mua")
    .show(100, truncate=False)
)

# 3. ĐỐI CHIẾU CUSTOMER-LEVEL: CHỈ LẦN MUA ĐẦU


first_purchase_validation = (
    snapshots
    .filter((F.col("purchase_sequence") == 1) & F.col("churn_180").isNotNull())
    .agg(
        F.count("*").alias("khach_du_180_ngay"),
        F.sum(F.when(F.col("churn_180") == 0, 1).otherwise(0)).alias("mua_lai_trong_180"),
        F.sum(F.when(F.col("churn_180") == 1, 1).otherwise(0)).alias("khong_mua_lai_180")
    )
    .withColumn(
        "ty_le_churn_180_pct",
        F.round(100 * F.col("khong_mua_lai_180") / F.col("khach_du_180_ngay"), 2)
    )
)

print("ĐỐI CHIẾU LẦN MUA ĐẦU — CUSTOMER LEVEL")
first_purchase_validation.show(truncate=False)


# 4. KHÁCH CÓ CẢ NON-CHURN VÀ CHURN Ở CÁC LẦN MUA KHÁC NHAU


customer_label_mix = (
    snapshots
    .filter(F.col("churn_180").isNotNull())
    .groupBy("customer_unique_id")
    .agg(
        F.min("churn_180").alias("min_label"),
        F.max("churn_180").alias("max_label"),
        F.count("*").alias("so_event_co_nhan")
    )
)

both_label_customers = (
    customer_label_mix
    .filter((F.col("min_label") == 0) & (F.col("max_label") == 1))
    .count()
)

print("Số khách từng có cả non-churn và churn:", both_label_customers)


# 5. GIỮ TẬP CÓ NHÃN 180 NGÀY


labeled_180 = (
    snapshots
    .filter(F.col("churn_180").isNotNull())
    .cache()
)

label_distribution = (
    labeled_180
    .groupBy("churn_180")
    .agg(
        F.count("*").alias("so_event"),
        F.countDistinct("customer_unique_id").alias("so_khach")
    )
    .withColumn(
        "ty_le_event_pct",
        F.round(100 * F.col("so_event") / F.sum("so_event").over(Window.partitionBy()), 2)
    )
    .orderBy("churn_180")
)

print("PHÂN BỐ NHÃN 180 NGÀY Ở CẤP EVENT")
label_distribution.show(truncate=False)

(
    labeled_180
    .repartition(8, "snapshot_date")
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(snapshot_labeled_dir))
)

print("Đã lưu tập có nhãn 180 ngày:", snapshot_labeled_dir)

labeled_180.unpersist()
snapshots.unpersist()


TỔNG QUAN PURCHASE-EVENT SNAPSHOT
+-----------------+--------+-------------+---------+------------+
|so_event_snapshot|so_khach|non_churn_180|churn_180|censored_180|
+-----------------+--------+-------------+---------+------------+
|96478            |93358   |2605         |55767    |38106       |
+-----------------+--------+-------------+---------+------------+

TRẠNG THÁI NHÃN 180 NGÀY
+----------------+--------+--------+
|label_status_180|so_event|so_khach|
+----------------+--------+--------+
|CENSORED        |38106   |38106   |
|CHURN_GAP_GT_180|515     |510     |
|CHURN_NO_RETURN |55252   |55252   |
|NON_CHURN       |2605    |2345    |
+----------------+--------+--------+

PHÂN BỐ SỐ LẦN MUA TRÊN MỖI KHÁCH
+----------+--------+
|so_lan_mua|so_khach|
+----------+--------+
|1         |90557   |
|2         |2573    |
|3         |181     |
|4         |28      |
|5         |9       |
|6         |5       |
|7         |3       |
|9         |1       |
|15        |1       |
+----------+---

DataFrame[customer_unique_id: string, snapshot_order_id: string, snapshot_at: timestamp, delivered_at: string, estimated_delivery_at: string, purchase_sequence: int, previous_purchase_at: timestamp, next_purchase_at: timestamp, first_purchase_at: string, days_since_previous_purchase: double, gap_days: double, customer_tenure_days: double, avg_interpurchase_days_to_snapshot: double, max_interpurchase_days_to_snapshot: double, snapshot_date: date, followup_end_90: timestamp, has_full_followup_90: string, churn_90: int, label_status_90: string, followup_end_120: timestamp, has_full_followup_120: string, churn_120: int, label_status_120: string, followup_end_180: timestamp, has_full_followup_180: string, churn_180: int, label_status_180: string, followup_days_available: double]

## Kết quả của notebook

- `processed/olist_customer_purchase_event_snapshots`: toàn bộ purchase-event snapshot, gồm cả censored.
- `processed/olist_customer_purchase_event_snapshots_labeled_180`: chỉ các observation có nhãn `churn_180` xác định.
- Feature Engineering và temporal split được chuyển hoàn toàn sang `05_Feature_Engineering.ipynb`.
